In [1]:
%pip install -q tensorflow numpy matplotlib pillow scikit-learn opencv-python seaborn tqdm

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\Users\\singh\\OneDrive\\Desktop\\Ayush\\coding\\internship\\Aerial Object Classification & Detection\\.venv311\\Lib\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\lib\\security\\credentials\\gcp_service_account_identity\\gcp_service_account_identity_credentials.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



# Aerial Object Classification & Detection

This notebook implements the internship project: binary classification (Bird vs Drone) using a Custom CNN and Transfer Learning, plus optional notes for YOLOv8 object detection and Streamlit deployment.

Follow cells in order. The notebook auto-checks for required packages and can perform a small test run to validate cells quickly.

In [2]:
# 1) Import & setup
import sys, subprocess, importlib
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from PIL import Image
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    HAS_NUMPY = HAS_MATPLOTLIB = HAS_PIL = HAS_TF = HAS_SKLEARN = HAS_SEABORN = True
except Exception as e:
    print(f'Import error: {e}')
    HAS_NUMPY = HAS_MATPLOTLIB = HAS_PIL = HAS_TF = HAS_SKLEARN = HAS_SEABORN = False
    np = plt = Image = tf = keras = layers = classification_report = confusion_matrix = sns = None

print('Environment flags — numpy:', HAS_NUMPY, 'matplotlib:', HAS_MATPLOTLIB, 'PIL:', HAS_PIL, 'TF:', HAS_TF, 'sklearn:', HAS_SKLEARN, 'seaborn:', HAS_SEABORN)

Import error: No module named 'tensorflow.python'
Environment flags — numpy: False matplotlib: False PIL: False TF: False sklearn: False seaborn: False


In [3]:
# 2) Dataset paths and basic inspection
from pathlib import Path
base_dir = Path('classification_dataset')
train_dir = base_dir / 'train'
valid_dir = base_dir / 'valid'
test_dir = base_dir / 'test'
print('Base path:', base_dir.resolve())

def count_images(folder):
    counts = {}
    if not folder.exists():
        return counts
    for c in folder.iterdir():
        if c.is_dir():
            counts[c.name] = len(list(c.glob('*.jpg'))) + len(list(c.glob('*.jpeg'))) + len(list(c.glob('*.png')))
    return counts

train_counts = count_images(train_dir)
valid_counts = count_images(valid_dir)
test_counts = count_images(test_dir)
print('Train counts:', train_counts)
print('Valid counts:', valid_counts)
print('Test counts:', test_counts)

Base path: C:\Users\singh\OneDrive\Desktop\Ayush\coding\internship\Aerial Object Classification & Detection\classification_dataset
Train counts: {'bird': 1414, 'drone': 1248}
Valid counts: {'bird': 217, 'drone': 225}
Test counts: {'bird': 121, 'drone': 94, 'images': 224, 'labels': 0}


In [4]:
# 3) Visualize sample images from each class (up to 4 per class)
def show_samples(folder, max_per_class=4):
    fig, axes = plt.subplots(2, max_per_class, figsize=(4*max_per_class,8))
    classes = sorted([d.name for d in folder.iterdir() if d.is_dir()])
    for r,cls in enumerate(classes[:2]):
        cls_dir = folder/cls
        imgs = list(cls_dir.glob('*.jpg'))[:max_per_class]
        for c, img_path in enumerate(imgs):
            ax = axes[r, c] if max_per_class>1 else axes[r]
            img = Image.open(img_path).convert('RGB')
            ax.imshow(img)
            ax.set_title(f'{cls} - {img_path.name}')
            ax.axis('off')
    plt.tight_layout()

# Show from training folder if available
if train_dir.exists():
    show_samples(train_dir, max_per_class=4)
else:
    print('Train directory not found; skipping sample visualization')

AttributeError: 'NoneType' object has no attribute 'subplots'

## 4) Prepare TensorFlow datasets for classification

The code below builds `tf.data` datasets using `image_dataset_from_directory`. For a quick test-run we can limit the number of samples.

In [ ]:
# Parameters
IMG_SIZE = (224,224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
seed = 123

# Create datasets (will load from folder structure). If folders missing, these will error - we guard.
if train_dir.exists():
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(str(train_dir),
        labels='inferred', label_mode='int', image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=seed)
else:
    train_ds = None
if valid_dir.exists():
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(str(valid_dir),
        labels='inferred', label_mode='int', image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=seed)
else:
    val_ds = None
if test_dir.exists():
    test_ds = tf.keras.preprocessing.image_dataset_from_directory(str(test_dir),
        labels='inferred', label_mode='int', image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=seed)
else:
    test_ds = None

# Prefetch for performance
if train_ds is not None:
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
if val_ds is not None:
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
if test_ds is not None:
    test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print('Datasets prepared:', 'train' if train_ds else 'train missing', 'val' if val_ds else 'val missing', 'test' if test_ds else 'test missing')

## 5) Data Augmentation and Normalization Layers

Use Keras preprocessing layers so augmentation is part of the model graph.

In [ ]:
# Data augmentation and normalization (guarded if TF missing)
if HAS_TF:
    data_augmentation = keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ], name='data_augmentation')
    normalization_layer = layers.Rescaling(1./255)
    print('Augmentation and normalization layers created')
else:
    data_augmentation = None
    normalization_layer = None
    print('TensorFlow not available — skipping augmentation/normalization layer creation')

## 6) Custom CNN model (small)

A compact CNN with batch norm and dropout for baseline.

In [ ]:
# Custom CNN model (small) — guarded by HAS_TF
if HAS_TF:
    def build_custom_cnn(input_shape=(224,224,3), num_classes=2):
        inputs = keras.Input(shape=input_shape)
        x = data_augmentation(inputs)
        x = normalization_layer(x)
        x = layers.Conv2D(32,3,activation='relu',padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(64,3,activation='relu',padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(128,3,activation='relu',padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.4)(x)
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        model = keras.Model(inputs, outputs, name='custom_cnn')
        return model

    custom_model = build_custom_cnn()
    custom_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    custom_model.summary()
else:
    custom_model = None
    print('TensorFlow not available — skipping custom CNN creation')

## 7) Transfer Learning (MobileNetV2)

Load a pretrained MobileNetV2 and fine-tune the top layers.

In [ ]:
# Transfer Learning model — guarded by HAS_TF
if HAS_TF:
    def build_transfer_model(input_shape=(224,224,3), num_classes=2):
        base = keras.applications.MobileNetV2(include_top=False, input_shape=input_shape, weights='imagenet')
        base.trainable = False
        inputs = keras.Input(shape=input_shape)
        x = data_augmentation(inputs)
        x = normalization_layer(x)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.4)(x)
        outputs = layers.Dense(num_classes, activation='softmax')(x)
        model = keras.Model(inputs, outputs, name='mobilenetv2_ft')
        return model

    transfer_model = build_transfer_model()
    transfer_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    transfer_model.summary()
else:
    transfer_model = None
    print('TensorFlow not available — skipping transfer learning model creation')

## 8) Training (quick test run)

We run a short training for 1 epoch on a small subset to validate the training loop. Set `full_training=True` to run full training locally.

In [ ]:
# Training (quick test run) — guarded by HAS_TF and presence of datasets/models
full_training = False  # set True if you want to train fully (may take long)
EPOCHS = 1 if not full_training else 20

if not HAS_TF:
    print('TensorFlow not available — skipping training')
elif train_ds is None:
    print('No training data found; skipping training step')
else:
    if custom_model is None:
        print('Custom model not built; skipping training')
    else:
        # For quick validation, take small subset if not full_training
        ds_train = train_ds.take(100) if not full_training else train_ds
        ds_val = val_ds.take(50) if (val_ds is not None and not full_training) else val_ds
        callbacks = [
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint('best_custom_cnn.h5', save_best_only=True)
        ]
        print('Training custom CNN for', EPOCHS, 'epoch(s)')
        history = custom_model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, callbacks=callbacks)
        print('Done')

## 9) Evaluation on Test Set

Compute predictions and print classification report. If `test_ds` is missing this will be skipped.

In [ ]:
# Evaluation on test set — guarded by HAS_TF
if not HAS_TF:
    print('TensorFlow not available — skipping evaluation')
elif test_ds is None:
    print('No test dataset found; skipping evaluation')
else:
    if custom_model is None:
        print('Custom model not built; skipping evaluation')
    else:
        # Collect true labels and predictions (use small subset for quick run)
        ds_eval = test_ds.take(100)
        y_true = []
        y_pred = []
        for x_batch, y_batch in ds_eval:
            preds = custom_model.predict(x_batch)
            preds_labels = np.argmax(preds, axis=1)
            y_true.extend(y_batch.numpy().tolist())
            y_pred.extend(preds_labels.tolist())
        if HAS_SKLEARN:
            print('Classification report (custom CNN):')
            print(classification_report(y_true, y_pred, digits=4))
        else:
            print('scikit-learn not available — cannot print classification report')

## 10) YOLOv8 (Optional)

Instructions to prepare and train YOLOv8 on `object_detection_Dataset`. This cell only contains installation and configuration commands; it does not start long training by default.

In [ ]:
# YOLOv8 optional install instructions (uncomment to run)
# Note: training YOLOv8 requires the `ultralytics` package and may need GPU/long time.
# try:
#     pip_install(['ultralytics'])
#     from ultralytics import YOLO
#     print('ultralytics installed')
# except Exception as e:
#     print('ultralytics install skipped or failed:', e)

print('YOLOv8 instructions: prepare a data.yaml with paths and class names, then call:')
print('!yolo detect train data=data.yaml model=yolov8n.pt epochs=50 imgsz=640')

## 11) Streamlit App (Deployment Notes)

A simple `app.py` is provided in a following cell. Run with: `streamlit run app.py`. The app loads a saved classification model and shows predictions.

In [ ]:
# Create a minimal Streamlit app file `app.py` in workspace
app_code = r'''
import streamlit as st
from PIL import Image
import numpy as np
import tensorflow as tf

st.title('Aerial Object Classification (Bird vs Drone)')
uploaded = st.file_uploader('Upload an image', type=['jpg','jpeg','png'])
if uploaded is not None:
    img = Image.open(uploaded).convert('RGB')
    st.image(img, caption='Uploaded image', use_column_width=True)
    # Preprocess
    img_resized = img.resize((224,224))
    x = np.array(img_resized) / 255.0
    x = np.expand_dims(x, 0)
    try:
        model = tf.keras.models.load_model('best_custom_cnn.h5')
        pred = model.predict(x)
        cls = np.argmax(pred, axis=1)[0]
        conf = float(np.max(pred))
        labels = ['bird','drone']
        st.success(f'Prediction: {labels[cls]} (confidence: {conf:.3f})')
    except Exception as e:
        st.warning('Model not found. Train and save a model as `best_custom_cnn.h5` in the workspace.\n', e)
'''
with open('app.py','w',encoding='utf-8') as f:
    f.write(app_code)
print('Wrote `app.py` — run with: streamlit run app.py')

---

### Notes and Next Steps
- To run full training, set `full_training = True` and re-run the training cell.
- For YOLOv8 detection training, prepare `data.yaml` in the `object_detection_Dataset` root and use `ultralytics` as shown.
- You can customize augmentations, model sizes, and training hyperparameters for better performance.

This notebook performs a short validation run by default so cells execute quickly and without errors. Increase compute for full experiments.